In [12]:

import copy
import pickle
import numpy as np

spin=3
f=open("gdstate,length=200,J2=1,Bond dimension=250,spin=3","rb")
M=pickle.load(f)
M=M[0]
L=200






def dmrg_mpo_j1j2(L,J1,J2,h,spin):
    W=[]
    
    
    if spin==2:
        I=np.array([[1,0],[0,1]])
        Spl=np.array([[0,1],[0,0]])
        Smi=np.array([[0,0],[1,0]])
        Sz=(1/2)*np.array([[1,0],[0,-1]])
        o=np.array([[0,0],[0,0]])
        
    if spin==3:
        I=np.array([[1,0,0],[0,1,0],[0,0,1]])
        
        Spl=np.sqrt(2)*np.array([[0,1,0],[0,0,1],[0,0,0]])
        Smi=np.sqrt(2)*np.array([[0,0,0],[1,0,0],[0,1,0]])
        Sz=np.array([[1,0,0],[0,0,0],[0,0,-1]])
        o=np.array([[0,0,0],[0,0,0],[0,0,0]])
        
    if spin==4:
        
        I=np.array([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,1]])
        Spl=np.array([[0,np.sqrt(3),0,0],[0,0,2,0],[0,0,0,np.sqrt(3)],[0,0,0,0]])
        Smi=np.array([[0,0,0,0],[np.sqrt(3),0,0,0],[0,2,0,0],[0,0,np.sqrt(3),0]])
        Sz=np.array([[3/2,0,0,0],[0,1/2,0,0],[0,0,-1/2,0],[0,0,0,-3/2]])
        o=np.array([[0,0,0,0],[0,0,0,0],[0,0,0,0],[0,0,0,0]])
        
    if spin==5:
        
        I=np.array([[1,0,0,0,0],[0,1,0,0,0],[0,0,1,0,0],[0,0,0,1,0],[0,0,0,0,1]])
        Spl=np.array([[0,2,0,0,0],[0,0,np.sqrt(6),0,0],[0,0,0,np.sqrt(6),0],[0,0,0,0,2],[0,0,0,0,0]])
        Smi=np.array([[0,0,0,0,0],[2,0,0,0,0],[0,np.sqrt(6),0,0,0],[0,0,np.sqrt(6),0,0],[0,0,0,2,0]])
        Sz=np.array([[2,0,0,0,0],[0,1,0,0,0],[0,0,0,0,0],[0,0,0,-1,0],[0,0,0,0,-2]])
        o=np.array([[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0]])
    
    if spin==6:
        
        I=np.array([[1,0,0,0,0,0],[0,1,0,0,0,0],[0,0,1,0,0,0],[0,0,0,1,0,0],[0,0,0,0,1,0],[0,0,0,0,0,1]])
        Spl=np.array([[0,np.sqrt(5),0,0,0,0],[0,0,np.sqrt(8),0,0,0],[0,0,0,np.sqrt(9),0,0],[0,0,0,0,np.sqrt(8),0],[0,0,0,0,0,np.sqrt(5)],[0,0,0,0,0,0]])
        Smi=np.array([[0,0,0,0,0,0],[np.sqrt(5),0,0,0,0,0],[0,np.sqrt(8),0,0,0,0],[0,0,np.sqrt(9),0,0,0],[0,0,0,np.sqrt(8),0,0],[0,0,0,0,np.sqrt(5),0]])
        Sz=np.array([[5/2,0,0,0,0,0],[0,3/2,0,0,0,0],[0,0,1/2,0,0,0],[0,0,0,-1/2,0,0],[0,0,0,0,-3/2,0],[0,0,0,0,0,-5/2]])
        o=np.array([[0,0,0,0,0,0],[0,0,0,0,0,0],[0,0,0,0,0,0],[0,0,0,0,0,0],[0,0,0,0,0,0],[0,0,0,0,0,0]])
        
    
    W0=np.hstack((-h*Sz,(J1/2)*Smi,(J1/2)*Spl,J1*Sz,(J2/2)*Smi,(J2/2)*Spl,J2*Sz,I))
    
    Wa=np.hstack((I,o,o,o,o,o,o,o))
    Wb=np.hstack((Spl,o,o,o,o,o,o,o))
    Wc=np.hstack((Smi,o,o,o,o,o,o,o))
    Wd=np.hstack((Sz,o,o,o,o,o,o,o))
    
    We=np.hstack((o,I,o,o,o,o,o,o))
   
    Wf=np.hstack((o,o,I,o,o,o,o,o))
    
    Wg=np.hstack((o,o,o,I,o,o,o,o))
    Wh=np.hstack((-h*Sz,(J1/2)*Smi,(J1/2)*Spl,J1*Sz,(J2/2)*Smi,(J2/2)*Spl,J2*Sz,I))
    
    W1=np.vstack((Wa,Wb,Wc,Wd,We,Wf,Wg,Wh))
    
    W2=np.vstack((I,Spl,Smi,Sz,o,o,o,-h*Sz))
    
    W0=W0.reshape(spin,8,spin)
    W0=np.transpose(W0,[0,2,1])
    
    W1=W1.reshape(8,spin,8,spin)
    W1=np.transpose(W1,[1,3,0,2])
    
    W2=W2.reshape(8,spin,spin)
    W2=np.transpose(W2,[1,2,0])
    
    
    W.append(W0)
    
    for i in range(L-2):
        W.append(W1)
     
    
    W.append(W2)
    
    return W





def dmrg_mpo_prod_prod(W,M1,M2):
    
        #W=MPOs,M=MPSs
        #Iteratively performing the products at the indvidual sites of MPOs and MPSs and
        # multiplying the new tensor with the existing chain of tensors
        #n is the site number which starts from zero and goes upto N-1.
        #N-1 is the last site number
        #N is the number of sites
        
        N=len(M1)
        L=len(M1)
            
        #First complete the left side
        prod_L=1
        prod_R=1
        
        for i in range(N):
        
            if i==0:
            
                prod1=np.tensordot(M1[i],W[i],axes=([0],[0]))#first MPS and MPO multiplied together
                prodin=np.tensordot(prod1,W[i],axes=([1],[0]))
                prod2=np.tensordot(prodin,M2[i].conj(),axes=([2],[0]))
                prod_L=prod2
                
            if i>0 and i<N-1:
            
                prod1=np.tensordot(prod_L,M1[i],axes=([0],[1]))#first MPS and MPO multiplied together
                
                prodin=np.tensordot(prod1,W[i],axes=([0,3],[2,0]))
                
                prod2=np.tensordot(prodin,W[i],axes=([0,3],[2,0]))
        
                prod_L=np.tensordot(prod2,M2[i].conj(),axes=([0,3],[1,0]))
                
    
            
            if i==N-1:
            
                prod1=np.tensordot(prod_L,M1[i],axes=([0],[1]))#first MPS and MPO multiplied together
                
                prodin=np.tensordot(prod1,W[i],axes=([0,3],[2,0]))
                
                prod2=np.tensordot(prodin,W[i],axes=([0,2],[2,0]))
            
            
              
                prod_L=np.tensordot(prod2,M2[i].conj(),axes=([0,1],[1,0]))        
    

              
    
                
        return prod_L
    
def dmrg_prod(W,M1,M2):
    
        #W=MPOs,M=MPSs
        #Iteratively performing the products at the indvidual sites of MPOs and MPSs and
        # multiplying the new tensor with the existing chain of tensors
        #n is the site number which starts from zero and goes upto N-1.
        #N-1 is the last site number
        #N is the number of sites
        
        N=len(M1)
        L=len(M1)
            
        #First complete the left side
        prod_L=1
        prod_R=1
        
        for i in range(N):
        
            if i==0:
            
                prod1=np.tensordot(M1[i],W[i],axes=([0],[0]))#first MPS and MPO multiplied together
                prod2=np.tensordot(prod1,M2[i].conj(),axes=([1],[0]))
                prod_L=prod2
                
            if i>0 and i<N-1:
            
                prod1=np.tensordot(prod_L,M1[i],axes=([0],[1]))#first MPS and MPO multiplied together
                prod2=np.tensordot(prod1,W[i],axes=([0,2],[2,0]))
        
                prod_L=np.tensordot(prod2,M2[i].conj(),axes=([0,2],[1,0]))
                
    
            
            if i==N-1:
            
                prod1=np.tensordot(prod_L,M1[i],axes=([0],[1]))#first MPS and MPO multiplied together
                prod2=np.tensordot(prod1,W[i],axes=([0,2],[2,0]))
            
            
              
                prod_L=np.tensordot(prod2,M2[i].conj(),axes=([0,1],[1,0]))        
    

              
    
                
        return prod_L
    
    
def dmrg_contract2(M1,M2):
    
        N=len(M1)
    
            
        #First complete the left side
        prod_L=1
        
        
        for i in range(N):
        
            if i==0:
            
                prod_L=np.tensordot(M1[i].conj(),M2[i],axes=([0],[0]))#first MPS and MPO multiplied together
               
                
            
                
            if i>0 and i<N-1:
            
                prod_L=np.tensordot(prod_L,M1[i].conj(),axes=([0],[1]))#first MPS and MPO multiplied together
            
        
                prod_L=np.tensordot(prod_L,M2[i],axes=([0,1],[1,0]))
                
               
            
            if i==N-1:
            
                prod_L=np.tensordot(prod_L,M1[i].conj(),axes=([0],[1]))#first MPS and MPO multiplied together
            
            
            
              
                prod_L=np.tensordot(prod_L,M2[i],axes=([0,1],[1,0]))        
    

              
               
        return prod_L



#mpo=dmrg_mpo_new(L,1,1,0.5,spin)
mpo=dmrg_mpo_j1j2(L,1,1,0.0,spin)
a=dmrg_mpo_prod_prod(mpo,M,M)
b=dmrg_prod(mpo,M,M)
print(dmrg_contract2(M,M))
print(a-b**2)
print(b/L)

#print(dmrg_contract(M1,M1))
#print(dmrg_contract2(M1,M1))

1.0000000000000124
0.005527749104658142
-1.5178629550655558
